# 06_class_thumbs_crop

원본 Colab 셀에서 분리. (`#@title 56종 대표 알약 크롭 (train 박스 기준) — 단독 실행 전체본`)


In [ ]:
#@title [매 세션 1] rfdetr 설치
#@markdown 런타임이 끊기면 설치된 패키지가 전부 사라지므로, 매 세션 rfdetr을 다시 설치해야 함
!pip install -q "rfdetr[train,loggers]"                  # RF-DETR 학습(train)+로깅(loggers) 의존성

In [ ]:
#@title [매 세션 2] 드라이브 마운트 + 경로 자동 탐색
#@markdown 세션마다 드라이브 연결이 끊기므로 재마운트 필요. 데이터·zip이 전부 드라이브에 있어 경로부터 다시 잡아야 함
from google.colab import drive                           # 코랩↔드라이브 연결 도구
drive.mount('/content/drive')                             # 드라이브 마운트

import os, glob                                          # 경로·탐색 도구
CANDS = [                                                # 흔한 후보 경로 먼저
    '/content/drive/MyDrive/1팀 공유 문서/ai12-level1-project/sprint_ai_project1_data',
    '/content/drive/MyDrive/ai12-level1-project/sprint_ai_project1_data',
]
DATA_ROOT = next((c for c in CANDS if os.path.exists(c)), None)   # 존재하는 첫 경로 채택
if DATA_ROOT is None:                                    # 후보에 없으면 전체 재귀 검색
    hits = glob.glob('/content/drive/MyDrive/**/sprint_ai_project1_data', recursive=True)
    DATA_ROOT = hits[0] if hits else None
assert DATA_ROOT, "sprint_ai_project1_data를 못 찾음"     # 못 찾으면 중단
PROJ_ROOT = os.path.dirname(DATA_ROOT)                   # zip·백업 공통 상위 경로

TEST_IMG = os.path.join(DATA_ROOT, 'test_images')        # 제출용 842장(추론 때 사용)
print("DATA_ROOT:", DATA_ROOT)                           # 채택 경로 확인
print("PROJ_ROOT:", PROJ_ROOT)

In [ ]:
#@title 56종 대표 알약 크롭 (train 박스 기준) — 단독 실행 전체본
#@markdown train_annotations에서 원본 56종 K코드별 첫 박스로 알약만 크롭 → 그리드 표시 + class_thumbs_74 폴더에 개별 저장. 18종(수집분)은 제외

!apt-get install -y fonts-nanum -qq                       # 한글 폰트(제목 깨짐 방지)
import matplotlib.font_manager as fm, matplotlib.pyplot as plt
fm.fontManager.addfont("/usr/share/fonts/truetype/nanum/NanumGothic.ttf")
plt.rcParams["font.family"] = "NanumGothic"               # 제목 한글 폰트
plt.rcParams["axes.unicode_minus"] = False

import os, re, json                                        # 경로·정규식·json 도구
from pathlib import Path
from PIL import Image                                      # 이미지 열기·크롭 도구
import pandas as pd
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

# ── [1] 경로 + 이름표 로드 ──────────────────────────────
TEAM_ROOT = Path("/content/drive/MyDrive/1팀 공유 문서")
USER_ROOT = TEAM_ROOT / "김태윤"
DATA_ROOT = TEAM_ROOT / "ai12-level1-project" / "sprint_ai_project1_data"  # 이미지 뿌리
ANN_ROOT  = DATA_ROOT / "train_annotations"               # 원본 56종 주석 뿌리
CSV = USER_ROOT / "class_name_map_74.csv"                 # 74종 이름표
OUT_DIR = USER_ROOT / "class_thumbs_74"; OUT_DIR.mkdir(exist_ok=True)  # 크롭 저장 폴더

names = pd.read_csv(CSV, dtype={"K코드": int})             # internal_id | K코드 | 약이름 | 파일명스템

def safe(name):                                           # 파일명 못 쓰는 문자 치환 도구
    return re.sub(r'[/\\:*?"<>|]', "-", str(name))        # / \ : * ? " < > | → -

# ── [2] train_annotations 훑어 K코드별 "첫 등장" (파일명, 박스) ──
first = {}                                                 # {K코드: (file_name, [x,y,w,h])}
scanned = 0
for jp in ANN_ROOT.rglob("*.json"):                       # 조합 폴더까지 전부 순회
    try: d = json.load(open(jp, encoding="utf-8"))
    except Exception: continue
    if not isinstance(d, dict) or "annotations" not in d: continue
    scanned += 1
    id2fn = {im["id"]: im["file_name"] for im in d.get("images", [])}  # image_id -> 파일명
    for a in d["annotations"]:
        k = int(a["category_id"])                          # 박스의 K코드
        if k not in first and a.get("image_id") in id2fn:  # 아직 없는 클래스면
            first[k] = (id2fn[a["image_id"]], a["bbox"])   # 첫 등장 기록
print(f"훑은 json: {scanned} / 박스 확보: {len(first)} 클래스 (56이면 정상)")

# ── [3] 파일명 -> 실제 경로 색인 (한 번만 스캔) ─────────────
fn2path = {}                                               # {파일명: 전체경로}
for ext in ("*.png", "*.jpg", "*.jpeg"):
    for p in DATA_ROOT.rglob(ext):                         # 이미지 뿌리 전체
        fn2path.setdefault(p.name, p)                      # 같은 이름은 먼저 잡힌 것 유지
print(f"이미지 색인: {len(fn2path)}장")

# ── [4] 크롭 + 그리드 표시 + 저장 (56종만) ─────────────────
have = names[names["K코드"].isin(first.keys())].reset_index(drop=True)  # 박스 있는 56행만
PAD = 8                                                    # 박스 주변 여백(px)
cols = 8; rows_n = (len(have) + cols - 1) // cols          # 8열, 행수는 자동
fig, axes = plt.subplots(rows_n, cols, figsize=(cols*2.2, rows_n*2.4))
axes = axes.ravel()

saved = 0
for idx, (_, r) in enumerate(have.iterrows()):            # 56종 순회
    ax = axes[idx]; ax.axis("off")
    k = int(r["K코드"]); iid = int(r["internal_id"])
    stem = safe(r["파일명스템"])                           # 금지문자 치환한 스템
    ax.set_title(f"{iid:02d} {r['약이름']}", fontsize=7)   # 제목: 01 보령부스파정5mg

    fn, (x, y, w, h) = first[k]                            # 첫 등장 파일명·박스
    path = fn2path.get(fn) or fn2path.get(Path(fn).name)   # 실제 경로 조회
    if path is None:
        ax.text(0.5, 0.5, "이미지\n못찾음", ha="center", va="center", fontsize=8, color="red"); continue
    try:
        img = Image.open(path).convert("RGB"); W, H = img.size  # 원본 열기
        box = (max(0, x-PAD), max(0, y-PAD), min(W, x+w+PAD), min(H, y+h+PAD))  # 여백 준 영역
        crop = img.crop(box)                               # 알약만 크롭
        crop.save(OUT_DIR / f"{stem}.png")                 # 안전한 이름으로 저장
        ax.imshow(crop); saved += 1                        # 그리드 표시
    except Exception:
        ax.text(0.5, 0.5, "에러", ha="center", va="center", fontsize=8, color="red")

for j in range(len(have), len(axes)): axes[j].axis("off")  # 남는 칸 숨김
plt.tight_layout(); plt.show()
print(f"\n저장 완료: {OUT_DIR} (크롭 {saved}종)")